<a href="https://colab.research.google.com/github/Sumit-Pathrabe/Ai_Model_for_Tech_and_Finance/blob/main/Ai_Model_for_Tech_and_Finance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install Unsloth and specific library versions to prevent version conflicts
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-7yjx4gq8/unsloth_f84e20f56547410197792da5f444eb74
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-7yjx4gq8/unsloth_f84e20f56547410197792da5f444eb74
  Resolved https://github.com/unslothai/unsloth.git to commit af6504f900fe611a056e66eec6ab74976eab7f34
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [5]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # Plenty of space for document summaries
dtype = None          # Auto-detects Colab's GPU setup
load_in_4bit = True   # Keeps the model lightweight and fast

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct", # Corrected model name
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    trust_remote_code = True,
)

# Apply PEFT/LoRA to only train 1% of the model's weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Model loaded and optimized successfully!")

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.5.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model loaded and optimized successfully!


In [ ]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Structuring your exact requirements: Tech/Finance jargon, professional answers, and summaries
data = [
    # Task 1: Understand Technical Jargon
    {"instruction": "Define what a Kubernetes Pod is and explain its relation to Docker containers.",
     "output": "A Pod is the smallest deployable unit in Kubernetes, representing a single instance of a running process. A Pod wraps one or more Docker containers, sharing networking, storage, and a unique IP address space, allowing tightly coupled containers to scale together efficiently."},

    # Task 2: Understand Financial Jargon
    {"instruction": "What is Equity Dilution and how does it happen in startups?",
     "output": "Equity dilution occurs when a company issues new shares of stock, reducing the ownership percentage of existing shareholders. This commonly occurs during venture capital funding rounds, stock option exercises for employees, or secondary public offerings."},

    # Task 3: Answer Industry Questions Accurately
    {"instruction": "Can you calculate the P/E Ratio of a company with a stock price of $150 and EPS of $5? Explain what this implies professionally.",
     "output": "The Price-to-Earnings (P/E) Ratio is calculated as Stock Price / Earnings Per Share (EPS). Here, $150 / $5 = 30. Professionally, a P/E of 30 means investors are willing to pay $30 for every $1 of current earnings, usually signaling high expectations for future enterprise growth or industry-specific tech scaling."},

    # Task 4: Summarize Technical Documents
    {"instruction": "Summarize the following release notes document:\n'We have migrated our REST APIs to native Microservices managed via Kubernetes. All deployment strategies now execute automatically via our new CI/CD pipeline, lowering infrastructure costs via optimized Docker container configurations.'",
     "output": "Summary: The technical infrastructure has completed a modernization migration moving legacy REST APIs into a Kubernetes-managed microservices architecture. Software releases are fully automated using a CI/CD pipeline, resulting in lowered infrastructure overhead thanks to optimized Docker resource utilization."},

    # Task 5: Summarize Financial Documents & Generate Professional Responses
    {"instruction": "Summarize this quarterly financial report excerpt:\n'Our EBITDA surged by 15% due to aggressive scale in our tech vertical, offsetting a brief dip in traditional asset growth. The balance sheet analysis indicates strong cash retention, paving the way for systematic SIP investments next fiscal year.'",
     "output": "Summary: The company reported a 15% growth in EBITDA driven primarily by its technology vertical, which countered slower traditional business segments. Due to a healthy balance sheet and robust liquid cash retention, management plans to initiate capital deployments via Systematic Investment Plans (SIP) next fiscal year."}
]

# Apply the formal Llama-3 structural prompt format
tokenizer = get_chat_template(tokenizer, chat_template = "llama-3")

def format_prompts(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{output}<|eot_id|>"
        texts.append(text)
    return { "text" : texts }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompts, batched = True)
print("Dataset compiled and mapped.")
